# Create GDX for GAMS Model

Given a year, we create data to be used by the GAMS model

Components that are ready for use in the model:
- capacities from parse_capacities
- prices from parse_prices_entsoe_sftp
- gas prices from parse_gas_prices_eurostat
- load from parse_load_entsoe_sftp and parse_load_eurostat for scaling (CH is assigned scaling of 1 since it is missing from Eurostat)
- NTC values from parse_transfer_capacity
- annual and hourly generation from parse_generation_Eurostat and parse_generation_entsoe_sftp
    - Eurostat is missing data for some countries and technologies so we replace them with ENTSO-E TP values
- hourly RES generation from parse_res_ninja (profiles) generation above
- hourly ror generation from parse_hydro_JRC
- storage and pump_storage inflows from parse_hydro_JRC as well as reservoir inflows calculated based on parse_entsoe_reservoir_level_sftp and parse_generation_entsoe_sftp since Norway 2017 values are missing from the prior source
- hydro reservoir levels from parse_reservoir_level_entsoe_sftp and supplemented with technology shares from parse_hydro_JRC(not using JRC levels from parse_hydro_ENTSO-E as they are identical for all years and only given for a few countries), countries that do not have a value assigned, i.e. start storage is going to be 0 but they get natural inflow over the year: 'BE', 'CZ', 'DE', 'DK', 'EE', 'GB', 'HU', 'IE', 'LU', 'NL', 'PL', 'SK'
- cost resource curves from Cost_resource_curves
- CHP values from parse_CHP_data
- monthly availability of peakload units based on entsoe outage data from parse_outage_entsoe_sftp

Also creates a gdx for calibration with the following parameters:
- r_price(country,t)
- r_demand(country,t)
- r_generation(tech,country,t)
- TRADE(fromcountry,tocountry,t)

Future projects
- load: IEA (https://www.iea.org/reports/monthly-electricity-statistics) seems to have different monthly data than Eurostat, but only for some of our countries - however yearly sums are identical
- load: there are still a few na values for some countries in some hours - so far only backward fill for previous hour to not mess up weekly profiles

## Packages and options

In [12]:
import pandas as pd
import numpy as np
# import gdxtools as gt
# import gams
import os
import sys
import datetime
import calendar
# import qgrid

** Select base year ** 

In [13]:
baseyear = 2017

** Select countries (central or EU)** 

In [14]:
#countries_list = "Countries"
countries_list = "Countries_EU"

In [15]:
if countries_list == "Countries":
    countries_out = ""
if countries_list == "Countries_EU":
    countries_out = "_EU"
countries_out

'_EU'

** Dropbox export **

In [16]:
dropbox = 'no'

In [17]:
#file path assignments
fn_additional = "../additional_data.xlsx"
fn_price_gas = "../parsed_data/price_gas_yearly_Eurostat.csv"
fn_cap = "../parsed_data/capacities.csv"
fn_price = "../parsed_data/prices_"+str(baseyear)+"_hourly_entsoe.csv"
fn_load_entsoe = "../parsed_data/load_"+str(baseyear)+"_hourly_entsoe.csv"
fn_load_eurostat = "../parsed_data/load_yearly_Eurostat.csv"
fn_ntc = "../parsed_data/ntc.csv"
fn_trade = "../parsed_data/trade_"+str(baseyear)+"_hourly_entsoe.csv"
fn_res_ninja = "../parsed_data/res_ninja_profiles.csv"
fn_gen_eurostat = "../parsed_data/generation_monthly_eurostat.csv"

fn_gen_entsoe_weekly = '../parsed_data/generation_'+str(baseyear)+'_weekly_entsoe.csv'
fn_gen_entsoe_hourly = '../parsed_data/generation_'+str(baseyear)+'_hourly_entsoe.csv'
fn_avail_peak = '../parsed_data/avail_peakload_GU_'+str(baseyear)+'_monthly_entsoe.csv'

fn_cost_res = "../parsed_data/cost_resource_curves.csv"

fn_ror = "../parsed_data/hydro_ror_generation_hourly_ENTSO-E_adequacy.csv"
fn_reservoir_inflow = "../parsed_data/hydro_storage_inflows_weekly_ENTSO-E_adequacy.csv"
fn_reservoir_profile_TP = '../parsed_data/reservoir_level_'+str(baseyear)+'_hourly_entsoe_TP.csv'
fn_reservoir_profile_weekly_TP = '../parsed_data/reservoir_level_'+str(baseyear)+'_weekly_entsoe_TP.csv'
fn_reservoir_size = '../parsed_data/hydro_capacities_base_ENTSO-E_adequacy.csv'

fn_heat_dem = "../parsed_data/heat_demand.csv"
fn_chp_gen = "../parsed_data/chp_generation.csv"

#dir_gdx = "../../"
# we export directly to the models data folder
dir_gdx = os.path.normpath(os.getcwd() + os.sep + os.pardir+ os.sep + os.pardir) + os.sep + "model" + os.sep + "data"  + os.sep
dir_out = "../"

## Additional data and settings

Get additional from Excel and impose settings.

### Sets

Countries in model

In [18]:
df_countries= pd.read_excel(fn_additional, sheet_name=countries_list, index_col="Country")
countries = list(df_countries.index)

Trading partners for demand correction

In [19]:
df_countries_trade = pd.read_excel(fn_additional, sheet_name='Countries_EU_trade', index_col="Country")
countries_trade = list(df_countries_trade.index)

In [20]:
countries_EU_27 = list(pd.read_excel(fn_additional, sheet_name='Countries_EU_27', index_col="Country").index)

Technology sets

In [21]:
df_techs = pd.read_excel(fn_additional, sheet_name="Technologies")
storages = list(df_techs.Hydro.dropna().unique())
renewables = list(df_techs.Renewable.dropna().unique())
old_renewables = list(df_techs["Old Renewables"].dropna().unique())
all_renewables = renewables + old_renewables
conventionals = list(df_techs.Conventional.ffill().unique())
baseload = list(df_techs["Baseload"].dropna().unique())
fixed_feedin = list(df_techs["Fixed"].dropna().unique())
peakload = list(df_techs["peak"].dropna().unique())
technologies = storages + renewables + conventionals

Fuels

In [22]:
df_fuels = pd.read_excel(fn_additional, sheet_name="Fuels")
fuels = list(df_fuels["Main Fuel"].unique())

Add fuel cost per country as a separate data frame. 

In [23]:
df_fuels_2017 = pd.read_excel(fn_additional, sheet_name="Fuels_2017")
df_fuels_2017 = df_fuels_2017.fillna(0)
df_fuels_2017.head()

,country,Main Fuel,Price,Price_2
0,AT,Gas,26.41,0.000001
1,BE,Gas,0.00,0.000001
2,CZ,Gas,0.00,0.000001
3,DK,Gas,0.00,0.000001
4,FI,Gas,24.17,0.000001


We load separate gas prices

In [24]:
df_price_gas = pd.read_csv(fn_price_gas).set_index(['country','year'])
df_price_gas = df_price_gas[df_price_gas.index.get_level_values('year') == baseyear]
df_price_gas = df_price_gas.reset_index().drop(columns='year')
df_price_gas['Main Fuel'] = 'Gas'
df_price_gas.head()

,country,EUR_per_MWh,Main Fuel
0,AT,25.60,Gas
1,BA,35.05,Gas
2,BE,19.30,Gas
3,BG,17.30,Gas
4,CZ,23.60,Gas


In [27]:
df_fuels_2017 = df_fuels_2017.merge(df_price_gas,how='left',left_on=['country','Main Fuel'], right_on=['country','Main Fuel'])
df_fuels_2017 = df_fuels_2017[df_fuels_2017.country.isin(countries)]
df_fuels_2017.loc[df_fuels_2017['EUR_per_MWh'] > 0, 'Price'] = df_fuels_2017['EUR_per_MWh']
df_fuels_2017.head()

,country,Main Fuel,Price,Price_2,EUR_per_MWh_x,EUR_per_MWh_y,EUR_per_MWh
0,AT,Gas,25.6,0.000001,25.6,25.6,25.6
1,BE,Gas,19.3,0.000001,19.3,19.3,19.3
2,CZ,Gas,23.6,0.000001,23.6,23.6,23.6
3,DK,Gas,24.9,0.000001,24.9,24.9,24.9
4,FI,Gas,42.3,0.000001,42.3,42.3,42.3


Add variable O&M cost as a separate data frame:

In [28]:
df_OM_2017 = pd.read_excel(fn_additional, sheet_name="OM_2017")
df_OM_2017.head(1)

,country,Technology,variable_OM_Cost
0,AT,Biomass,2.6


### Mappings for missing data

Mapping for missing fuel prices

In [29]:
df_fuel_price_maps = pd.read_excel(fn_additional, sheet_name="FuelPriceAs", index_col = 0)

fuel_price_maps = []
for row in df_fuel_price_maps.iterrows():
    country = row[0]
    for fuel in df_fuel_price_maps.columns:
        fuel_price_as = row[1].loc[fuel]
        if pd.notnull(fuel_price_as):
            fuel_price_maps.append((country, fuel_price_as, fuel))

Mapping for missing variable O&M costs

In [30]:
df_om_maps = pd.read_excel(fn_additional, sheet_name="OMAs", index_col = 0)

om_maps = []
for row in df_om_maps.iterrows():
    country = row[0]
    for tech in df_om_maps.columns:
        om_as = row[1].loc[tech]
        if pd.notnull(om_as):
            om_maps.append((country, om_as, tech))

### Technology cost specifications

In [31]:
df_cost_in = pd.read_excel(fn_additional, sheet_name="Cost")
df_cost_in.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 13 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Main Fuel                                12 non-null     object 
 1   Technology                               15 non-null     object 
 2   Average Efficiency                       15 non-null     float64
 3   Availability                             15 non-null     float64
 4   Min Generation                           15 non-null     float64
 5   Startup Costs                            15 non-null     int64  
 6   Reserve                                  15 non-null     int64  
 7   variable OM Cost [Euro/MWh]              15 non-null     float64
 8   variable OM Cost quadratic [Euro/MWh^2]  15 non-null     float64
 9   LoadGradient                             15 non-null     float64
 10  RampingCost                              15 non-null

In [32]:
lst_df = []
for c in countries:
    df = df_cost_in.copy()
    df["country"] = c
    lst_df.append(df)
df_cost = pd.concat(lst_df, sort=True)

Additional availability data for peak load technologies based on entsoe outage data

In [33]:
df_avail_peak = pd.read_csv(fn_avail_peak).set_index(['country','technology','Month'])
df_avail_peak = df_avail_peak[df_avail_peak.index.get_level_values('country').isin(countries)]
df_avail_peak = df_avail_peak[df_avail_peak.index.get_level_values('technology').isin(peakload)]
df_avail_peak.head()

avail
country technology Month          
AT      Gas        1      0.999792
                   2      0.996587
                   3      0.974291
                   4      0.999691
                   5      0.999265

## Capacities

Select country and year and make unique plants prefixing technologies by country name:

In [34]:
df_cap_in = pd.read_csv(fn_cap)
df_cap = df_cap_in[(df_cap_in.country.isin(countries)
                    & df_cap_in.technology.isin(technologies)
                    )]
df_cap = df_cap[df_cap.year == baseyear].drop('year',axis=1).set_index(['country','technology'])
#this does not seem to be used in final code but lets not delete it for now
#df_cap["plant"] = df_cap.country + "_" + df_cap.technology
df_cap.head()

capacity
country technology          
AT      Biomass        572.0
        Gas           4841.0
        HardCoal       598.0
        Oil            166.0
        Other          978.0

we load capacities from the new hydro data set

even though the baseyear is not given for ENTSO-E capacities, they seem to be fairly recent and much more comprehensive (covering more countries) than what we had before so we just use them

In [35]:
df_cap_hydro = pd.read_csv(fn_reservoir_size)
df_cap_hydro = df_cap_hydro.rename(columns={
    'Pump Storage - Closed Loop - Total turbining capacity (MW)':'PumpClosed',  
    'Pump Storage - Open Loop - Total turbining capacity (MW)':'PumpOpen',
#    'Reservoir - Total turbining capacity (MW)':'Reservoir',
#    'Run-of-River and pondage - Total turbining capacity (MW)':'RunOfRiver'
                                }).set_index('country')
df_cap_hydro = df_cap_hydro[['PumpClosed','PumpOpen']]
df_cap_hydro = pd.DataFrame(df_cap_hydro.stack()).reset_index()
df_cap_hydro.columns = ['country','technology','capacity']
df_cap_hydro = pd.DataFrame(df_cap_hydro.set_index(['country','technology'])['capacity'])
df_cap_hydro.head()

capacity
country technology          
AL      PumpClosed      0.00
        PumpOpen        0.00
AT      PumpClosed      0.00
        PumpOpen     3459.28
BA      PumpClosed      0.00

load pumping capacity as well and merge the two

In [36]:
df_cap_hydro_pump = pd.read_csv(fn_reservoir_size)
df_cap_hydro_pump = df_cap_hydro_pump.rename(columns={
    'Pump Storage - Closed Loop - Total pumping capacity (MW)':'PumpClosed',  
    'Pump Storage - Open Loop - Total pumping capacity (MW)':'PumpOpen'}
                                 ).set_index('country')
df_cap_hydro_pump = df_cap_hydro_pump[['PumpClosed','PumpOpen']]
df_cap_hydro_pump = pd.DataFrame(df_cap_hydro_pump.stack()).reset_index()
df_cap_hydro_pump.columns = ['country','technology','pumping']
df_cap_hydro_pump = pd.DataFrame(df_cap_hydro_pump.set_index(['country','technology'])['pumping'])
df_cap_hydro_pump = abs(df_cap_hydro_pump)
df_cap_hydro = df_cap_hydro.join(df_cap_hydro_pump,how='outer')
df_cap_hydro.head()

capacity   pumping
country technology                    
AL      PumpClosed      0.00     0.000
        PumpOpen        0.00     0.000
AT      PumpClosed      0.00     0.000
        PumpOpen     3459.28  2559.726
BA      PumpClosed      0.00     0.000

now join with entsoe

In [38]:
df_cap = pd.concat([df_cap, df_cap_hydro])
df_cap = df_cap.reset_index()
df_cap = df_cap[(df_cap.country.isin(countries)
                    & df_cap.technology.isin(technologies)
                    )]
df_cap = df_cap.set_index(['technology','country']).fillna(0)
df_cap = abs(df_cap)
df_cap.head()

,,capacity,pumping
technology,country,,
Biomass,AT,572.0,0.0
Gas,AT,4841.0,0.0
HardCoal,AT,598.0,0.0
Oil,AT,166.0,0.0
Other,AT,978.0,0.0


## Generation for all technologies

load monthly values from eurostat

In [41]:
df_monthly_generation_in =  pd.read_csv(fn_gen_eurostat)
df_monthly_generation_in = df_monthly_generation_in[df_monthly_generation_in.year == baseyear]
df_monthly_generation = df_monthly_generation_in[df_monthly_generation_in.country.isin(countries)]
df_monthly_generation = df_monthly_generation.rename(columns={'tech':'technology'})
df_monthly_generation = df_monthly_generation[df_monthly_generation['MWh'] > 0]
df_monthly_generation.head(1)

,country,time,technology,year,month,MWh
0,AT,2017M01,Biomass,2017,1,197544.0


create yearly eurostat df

In [42]:
df_eurostat_year = df_monthly_generation.groupby(['year','country','technology']).sum().reset_index().drop(['time','year','month'], axis=1).set_index(['country','technology'])
df_eurostat_year = df_eurostat_year.rename(columns={'MWh':'eurostat'})
df_eurostat_year.head(1)

,,eurostat
country,technology,
AT,Biomass,2510588.0


also load entsoe TP hourly data

In [43]:
df_hourly_generation_entsoe_in =  pd.read_csv(fn_gen_entsoe_hourly,parse_dates=True, index_col="date").reset_index()
df_hourly_entsoe = df_hourly_generation_entsoe_in.rename(columns={'tech':'technology','date':'time','net_generation':'MWh_hourly'})
df_hourly_entsoe = df_hourly_entsoe.set_index(['country','technology','time'])[['MWh_hourly']]
df_hourly_entsoe.head(1)

,,,MWh_hourly
country,technology,time,
AT,Biomass,2017-01-01,300.0


In [44]:
df_hourly_entsoe_test = df_hourly_entsoe.copy()
df_hourly_entsoe_test['generator'] = df_hourly_entsoe_test.index.get_level_values('country') + df_hourly_entsoe_test.index.get_level_values('technology')
df_hourly_entsoe_test = df_hourly_entsoe_test.groupby('generator').sum()
df_hourly_entsoe_test = df_hourly_entsoe_test[df_hourly_entsoe_test.MWh_hourly > 0]
df_hourly_entsoe_test.head(1)

,MWh_hourly
generator,
ATBiomass,2600148.0


check for missing values in entsoe

In [45]:
df_eurostat_year_test = df_eurostat_year[df_eurostat_year.index.get_level_values('technology').isin(technologies)].copy()
df_eurostat_year_test['generator'] = df_eurostat_year_test.index.get_level_values('country') + df_eurostat_year_test.index.get_level_values('technology')

In [46]:
missing = np.setdiff1d(df_eurostat_year_test.generator.unique(),df_hourly_entsoe_test.index.get_level_values('generator').unique())
df_eurostat_year_test.eurostat[df_eurostat_year_test.generator.isin(missing)].sum() / df_eurostat_year_test.eurostat.sum() * 100

1.850511114310232

We manually fill missing thermal generation profiles with profiles from neighbouring countries, aggregate values will be set to eurostat

In [ ]:
df_hourly_entsoe_pivot = df_hourly_entsoe.pivot_table(index=['time','technology'],columns='country',values='MWh_hourly')

df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'AT'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'DE']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'BG'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'IT']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Biomass', 'GR'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Biomass', 'BG']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'GR'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'IT']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Other', 'GR'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Other', 'BG']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Biomass', 'IE'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Biomass', 'GB']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'NL'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'BE']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Biomass', 'NO'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Biomass', 'DK']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'PT'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'ES']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'RO'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'GR']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Biomass', 'SE'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Biomass', 'DK']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Gas', 'SE'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Gas', 'NO']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'SE'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'DK']

#HR and LU are completely missing
df_hourly_entsoe_pivot['HR'] = df_hourly_entsoe_pivot['SI']
df_hourly_entsoe_pivot['LU'] = df_hourly_entsoe_pivot['DE']
#HR is also missing HardCoal generation in entsoe data so we copy the profile from PL
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'HardCoal', 'HR'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'HardCoal', 'PL']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'HardCoal', 'SE'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'HardCoal', 'DK']

df_hourly_entsoe_filled = pd.DataFrame(df_hourly_entsoe_pivot.stack()).rename(columns={0:'MWh_hourly'}).reset_index().set_index(['country','technology','time'])
df_hourly_entsoe_filled.head(1)

,,,MWh_hourly
country,technology,time,
AT,Biomass,2017-01-01,300.0


create yearly aggregate

In [34]:
df_yearly_entsoe = df_hourly_entsoe_filled.reset_index().groupby(['country','technology']).sum()
df_yearly_entsoe = df_yearly_entsoe.rename(columns={'MWh_hourly':'entsoe'})
df_yearly_entsoe.head()

entsoe
country technology              
AT      Biomass     2.600148e+06
        Gas         9.727145e+06
        HardCoal    1.406821e+06
        Oil         1.887113e+06
        Other       1.069333e+06

merge the two yearly dfs and fill missing (or smaller) eurostat data with entsoe values 

In [35]:
df_generation_year = df_yearly_entsoe.join(df_eurostat_year, how='outer')
df_generation_year['MWh_year'] = df_generation_year['eurostat'].fillna(df_generation_year['entsoe'])
df_generation_year['MWh_year'][df_generation_year['entsoe'] > df_generation_year['eurostat']] = df_generation_year['entsoe']
#replace 0s from Eurostat with NaN, otherwise, we get inf values for scaling
df_generation_year['MWh_year'] = df_generation_year['MWh_year'].replace(0, np.nan)
df_generation_year.head()

entsoe    eurostat    MWh_year
country technology                                   
AT      Biomass     2600148.0   2510588.0   2600148.0
        Coal              NaN   3643473.0   3643473.0
        Gas         9727145.0  10117664.0  10117664.0
        HardCoal    1406820.7         NaN   1406820.7
        Hydro             NaN  38052344.0  38052344.0

Make sure that filled values doe not mess with scaling

In [36]:
df_generation_year['MWh_year'][((df_generation_year.index.get_level_values('country') == 'AT') & (df_generation_year.index.get_level_values('technology') == 'Oil'))] = df_generation_year['eurostat']
df_generation_year['MWh_year'][((df_generation_year.index.get_level_values('country') == 'BG') & (df_generation_year.index.get_level_values('technology') == 'Oil'))] = df_generation_year['eurostat']
df_generation_year['MWh_year'][((df_generation_year.index.get_level_values('country') == 'GR') & (df_generation_year.index.get_level_values('technology') == 'Biomass'))] = df_generation_year['eurostat']
df_generation_year['MWh_year'][((df_generation_year.index.get_level_values('country') == 'GR') & (df_generation_year.index.get_level_values('technology') == 'Oil'))] = df_generation_year['eurostat']
df_generation_year['MWh_year'][((df_generation_year.index.get_level_values('country') == 'GR') & (df_generation_year.index.get_level_values('technology') == 'Other'))] = df_generation_year['eurostat']
df_generation_year['MWh_year'][((df_generation_year.index.get_level_values('country') == 'HR') & (df_generation_year.index.get_level_values('technology') == 'Biomass'))] = df_generation_year['eurostat']
df_generation_year['MWh_year'][((df_generation_year.index.get_level_values('country') == 'HR') & (df_generation_year.index.get_level_values('technology') == 'Gas'))] = df_generation_year['eurostat']
df_generation_year['MWh_year'][((df_generation_year.index.get_level_values('country') == 'HR') & (df_generation_year.index.get_level_values('technology') == 'Oil'))] = df_generation_year['eurostat']
df_generation_year['MWh_year'][((df_generation_year.index.get_level_values('country') == 'IE') & (df_generation_year.index.get_level_values('technology') == 'Biomass'))] = df_generation_year['eurostat']
df_generation_year['MWh_year'][((df_generation_year.index.get_level_values('country') == 'LU') & (df_generation_year.index.get_level_values('technology') == 'Biomass'))] = df_generation_year['eurostat']
df_generation_year['MWh_year'][((df_generation_year.index.get_level_values('country') == 'LU') & (df_generation_year.index.get_level_values('technology') == 'Gas'))] = df_generation_year['eurostat']
df_generation_year['MWh_year'][((df_generation_year.index.get_level_values('country') == 'LU') & (df_generation_year.index.get_level_values('technology') == 'Other'))] = df_generation_year['eurostat']
df_generation_year['MWh_year'][((df_generation_year.index.get_level_values('country') == 'NL') & (df_generation_year.index.get_level_values('technology') == 'Oil'))] = df_generation_year['eurostat']
df_generation_year['MWh_year'][((df_generation_year.index.get_level_values('country') == 'NO') & (df_generation_year.index.get_level_values('technology') == 'Biomass'))] = df_generation_year['eurostat']
df_generation_year['MWh_year'][((df_generation_year.index.get_level_values('country') == 'PT') & (df_generation_year.index.get_level_values('technology') == 'Oil'))] = df_generation_year['eurostat']
df_generation_year['MWh_year'][((df_generation_year.index.get_level_values('country') == 'RO') & (df_generation_year.index.get_level_values('technology') == 'Oil'))] = df_generation_year['eurostat']
df_generation_year['MWh_year'][((df_generation_year.index.get_level_values('country') == 'SE') & (df_generation_year.index.get_level_values('technology') == 'Biomass'))] = df_generation_year['eurostat']
df_generation_year['MWh_year'][((df_generation_year.index.get_level_values('country') == 'SE') & (df_generation_year.index.get_level_values('technology') == 'Gas'))] = df_generation_year['eurostat']
df_generation_year['MWh_year'][((df_generation_year.index.get_level_values('country') == 'SE') & (df_generation_year.index.get_level_values('technology') == 'Oil'))] = df_generation_year['eurostat']

df_generation_year['MWh_year'][df_generation_year.index.get_level_values('country') == 'HR'] = df_generation_year['eurostat']
df_generation_year['MWh_year'][df_generation_year.index.get_level_values('country') == 'LU'] = df_generation_year['eurostat']
df_generation_year.head()

entsoe    eurostat    MWh_year
country technology                                   
AT      Biomass     2600148.0   2510588.0   2600148.0
        Coal              NaN   3643473.0   3643473.0
        Gas         9727145.0  10117664.0  10117664.0
        HardCoal    1406820.7         NaN   1406820.7
        Hydro             NaN  38052344.0  38052344.0

Eurostat only has coal but we can scale values in countries where only one of the two is present

In [37]:
df_generation_year.loc[('AT', 'HardCoal'), 'MWh_year'] = df_generation_year.loc[('AT', 'Coal'), 'eurostat']
df_generation_year.loc[('DK', 'HardCoal'), 'MWh_year'] = df_generation_year.loc[('DK', 'Coal'), 'eurostat']
df_generation_year.loc[('FI', 'HardCoal'), 'MWh_year'] = df_generation_year.loc[('FI', 'Coal'), 'eurostat']
df_generation_year.loc[('FR', 'HardCoal'), 'MWh_year'] = df_generation_year.loc[('FR', 'Coal'), 'eurostat']
df_generation_year.loc[('NL', 'HardCoal'), 'MWh_year'] = df_generation_year.loc[('NL', 'Coal'), 'eurostat']
df_generation_year.loc[('SE', 'HardCoal'), 'MWh_year'] = df_generation_year.loc[('SE', 'Coal'), 'eurostat']

df_generation_year.loc[('HR', 'HardCoal'), 'MWh_year'] = df_generation_year.loc[('HR', 'Coal'), 'eurostat']
df_generation_year.head()

entsoe    eurostat    MWh_year
country technology                                   
AT      Biomass     2600148.0   2510588.0   2600148.0
        Coal              NaN   3643473.0   3643473.0
        Gas         9727145.0  10117664.0  10117664.0
        HardCoal    1406820.7         NaN   3643473.0
        Hydro             NaN  38052344.0  38052344.0

Eventually, calculate scaling and drop not needed techs

In [38]:
df_generation_year['scale'] = df_generation_year['MWh_year']/df_generation_year['entsoe']
df_generation_year = df_generation_year[df_generation_year.index.get_level_values('technology').isin(technologies)]

In [39]:
df_generation_year.head()

entsoe    eurostat    MWh_year     scale
country technology                                                
AT      Biomass     2.600148e+06   2510588.0   2600148.0  1.000000
        Gas         9.727145e+06  10117664.0  10117664.0  1.040147
        HardCoal    1.406821e+06         NaN   3643473.0  2.589863
        Oil         1.887113e+06    722461.0    722461.0  0.382839
        Other       1.069333e+06   6696765.0   6696765.0  6.262562

Use calculated scaling to scale hourly entsoe values 

In [40]:
df_generation_hourly_1 = pd.DataFrame()
df_generation_hourly_1 = df_hourly_entsoe_filled.join(df_generation_year[['scale','MWh_year']],how='outer')
df_generation_hourly_1['MWh'] = df_generation_hourly_1['MWh_hourly'] * df_generation_hourly_1['scale']
df_generation_hourly_1 = df_generation_hourly_1[['MWh','MWh_year']]
df_generation_hourly_1.head()

MWh   MWh_year
country technology time                                 
AT      Biomass    2017-01-01 00:00:00  300.0  2600148.0
                   2017-01-01 01:00:00  300.0  2600148.0
                   2017-01-01 02:00:00  300.0  2600148.0
                   2017-01-01 03:00:00  300.0  2600148.0
                   2017-01-01 04:00:00  300.0  2600148.0

### Renewable generation

Get renewable generation for countries and base year

In [41]:
df_res_ninja_in = pd.read_csv(fn_res_ninja, parse_dates=True, index_col="time")
df_res_ninja = df_res_ninja_in[(df_res_ninja_in.country.isin(countries))
                     & (df_res_ninja_in.tech.isin(renewables)
                     & (pd.DatetimeIndex(df_res_ninja_in.index).year == baseyear)
                     )].copy().reset_index()

We normalize to hourly share of total yearly generation

In [42]:
df_res_ninja_year = df_res_ninja.groupby(['country','tech']).sum()
df_res_ninja_profile = df_res_ninja.merge(df_res_ninja_year,how='left',on=['country','tech'])
df_res_ninja_profile['profile'] = df_res_ninja_profile['capacity_factor_x']/df_res_ninja_profile['capacity_factor_y']
df_res_ninja_profile = df_res_ninja_profile.rename(columns={'tech':'technology'})
df_res = df_res_ninja_profile.set_index(['country','technology','time'])[['profile']]
df_res.head()

,,,profile
country,technology,time,
AT,WindOffshore,2017-01-01,0.000023
BE,WindOffshore,2017-01-01,0.000154
BG,WindOffshore,2017-01-01,0.000160
CH,WindOffshore,2017-01-01,0.000064
CZ,WindOffshore,2017-01-01,0.000117


now we add the profile to hourly generation

In [43]:
df_generation_hourly_2 = df_generation_hourly_1.join(df_res, how='outer')
df_generation_hourly_2['MWh'][df_generation_hourly_2['profile'].notna()] = df_generation_hourly_2['MWh_year'] * df_generation_hourly_2['profile']
df_generation_hourly_2.head()

MWh   MWh_year  profile
country technology time                                          
AT      Biomass    2017-01-01 00:00:00  300.0  2600148.0      NaN
                   2017-01-01 01:00:00  300.0  2600148.0      NaN
                   2017-01-01 02:00:00  300.0  2600148.0      NaN
                   2017-01-01 03:00:00  300.0  2600148.0      NaN
                   2017-01-01 04:00:00  300.0  2600148.0      NaN

### RoR Production

get production of Run-of-Rivers

In [44]:
df_ror_in = pd.read_csv(fn_ror, parse_dates=True, index_col="date")
df_ror_in = df_ror_in[df_ror_in.country.isin(countries)][str(baseyear)].copy()
df_ror_in = df_ror_in.reset_index()
df_ror_in.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8395 entries, 0 to 8394
Data columns (total 4 columns):
 #   Column                                        Non-Null Count  Dtype         
---  ------                                        --------------  -----         
 0   date                                          8395 non-null   datetime64[ns]
 1   country                                       8395 non-null   object        
 2   Run of River Hydro Generation in GWh per day  8395 non-null   float64       
 3   RoR generation MWh per hour                   8395 non-null   float64       
dtypes: datetime64[ns](1), float64(2), object(1)
memory usage: 262.5+ KB


C:\Users\JSAVEL~1\AppData\Local\Temp/ipykernel_11892/2849543057.py:2: FutureWarning: Indexing a DataFrame with a datetimelike index using a single string to slice the rows, like `frame[string]`, is deprecated and will be removed in a future version. Use `frame.loc[string]` instead.
  df_ror_in = df_ror_in[df_ror_in.country.isin(countries)][str(baseyear)].copy()


In [45]:
#resample to hourly values by pivoting and then stacking
df_ror = df_ror_in.pivot(index='date',columns='country')[['RoR generation MWh per hour']].resample('H').fillna("pad").stack()
df_ror['technology'] = 'RunOfRiver'
df_ror = df_ror.reset_index().rename(columns={'date':'time','RoR generation MWh per hour':'MWh_JRC'}).set_index(['country','technology','time'])
df_ror.reset_index().info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200951 entries, 0 to 200950
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   country     200951 non-null  object        
 1   technology  200951 non-null  object        
 2   time        200951 non-null  datetime64[ns]
 3   MWh_JRC     200951 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(2)
memory usage: 6.1+ MB


Also create a yearly df for comparison to entsoe-TP and eurostat

In [46]:
df_ror_year = df_ror.reset_index().groupby(['country','technology']).sum()
df_ror_year.head()

,,MWh_JRC
country,technology,
AT,RunOfRiver,2.862780e+07
BE,RunOfRiver,2.615921e+05
BG,RunOfRiver,9.718508e+05
CH,RunOfRiver,1.598404e+07
CZ,RunOfRiver,4.410963e+05


### merging all generation dfs

In [47]:
df_generation_year = df_generation_year.join(df_ror_year, how='outer')
df_generation_year['MWh_year'][df_generation_year['MWh_JRC'] > 0] = df_generation_year['MWh_JRC']
df_generation_year['scale'][df_generation_year['MWh_JRC'] > 0] = df_generation_year['MWh_JRC'] / df_generation_year['entsoe']
df_generation_year.to_csv(dir_out+'compare_gen_yearly.csv')

In [48]:
df_generation_year[df_generation_year.MWh_JRC > 0].head()

,,entsoe,eurostat,MWh_year,scale,MWh_JRC
country,technology,,,,,
AT,RunOfRiver,26857262.60,NaN,2.862780e+07,1.065924,2.862780e+07
BE,RunOfRiver,137920.07,NaN,2.615921e+05,1.896693,2.615921e+05
BG,RunOfRiver,269047.00,NaN,9.718508e+05,3.612197,9.718508e+05
CH,RunOfRiver,687037.67,NaN,1.598404e+07,23.265161,1.598404e+07
CZ,RunOfRiver,821062.25,NaN,4.410963e+05,0.537226,4.410963e+05


And create hourly df

In [49]:
df_generation_hourly_3 = df_generation_hourly_2.join(df_ror, how='outer')
df_generation_hourly_3.head()

MWh   MWh_year  profile  MWh_JRC
country technology time                                                   
AT      Biomass    2017-01-01 00:00:00  300.0  2600148.0      NaN      NaN
                   2017-01-01 01:00:00  300.0  2600148.0      NaN      NaN
                   2017-01-01 02:00:00  300.0  2600148.0      NaN      NaN
                   2017-01-01 03:00:00  300.0  2600148.0      NaN      NaN
                   2017-01-01 04:00:00  300.0  2600148.0      NaN      NaN

In [50]:
df_generation = df_generation_hourly_3.copy()
df_generation['MWh'][df_generation['MWh_JRC'] > 0] = df_generation['MWh_JRC']
df_generation = df_generation[['MWh','profile']].fillna(0)
df_generation = df_generation[df_generation.index.get_level_values('country').isin(countries)]
df_generation.head(1)

,,,MWh,profile
country,technology,time,,
AT,Biomass,2017-01-01,300.0,0.0


Also create hourly res and ror dfs for easier export

In [51]:
df_res = df_generation.copy()
df_res = df_res[df_res.index.get_level_values('technology').isin(renewables)]
df_res.head()

MWh  profile
country technology time                             
AT      Solar      2017-01-01 00:00:00  0.0      0.0
                   2017-01-01 01:00:00  0.0      0.0
                   2017-01-01 02:00:00  0.0      0.0
                   2017-01-01 03:00:00  0.0      0.0
                   2017-01-01 04:00:00  0.0      0.0

In [52]:
df_ror = df_generation.reset_index().copy()
df_ror = df_ror[df_ror.technology.isin({'RunOfRiver'})]
df_ror = df_ror.drop(['technology','profile'],axis=1).set_index(['country','time'])
df_ror.head()

MWh
country time                            
AT      2017-01-01 00:00:00  2123.500832
        2017-01-01 01:00:00  2123.500832
        2017-01-01 02:00:00  2123.500832
        2017-01-01 03:00:00  2123.500832
        2017-01-01 04:00:00  2123.500832

Drop profile from df_generation

In [53]:
df_generation = df_generation[['MWh']]
df_generation.head()

MWh
country technology time                      
AT      Biomass    2017-01-01 00:00:00  300.0
                   2017-01-01 01:00:00  300.0
                   2017-01-01 02:00:00  300.0
                   2017-01-01 03:00:00  300.0
                   2017-01-01 04:00:00  300.0

Create monthly generation df for availability calibration of baseload technologies

In [54]:
df_generation_monthly = df_generation.groupby([pd.Grouper(freq='M', level='time'), "country", "technology"]).sum().reset_index()
df_generation_monthly['month'] = pd.DatetimeIndex(df_generation_monthly['time']).month
df_generation_monthly = df_generation_monthly.set_index(["month", "country", "technology"])[['MWh']]
df_generation_monthly.head()

MWh
month country technology              
1     AT      Biomass     2.271770e+05
              Gas         2.308581e+06
              HardCoal    8.062415e+05
              Oil         1.057524e+05
              Other       5.687663e+05

Aggregate to annual values and compare to df_generation_year

In [55]:
df_annual = df_generation.groupby(["country", "technology"]).sum()
df_annual.head()

MWh
country technology            
AT      Biomass      2600148.0
        Gas         10117664.0
        HardCoal     3643473.0
        Oil           722461.0
        Other        6696765.0

In [56]:
df_generation_year_compare = df_generation_year[['MWh_year']]
df_compare = df_annual.join(df_generation_year_compare, how='outer')
df_compare['diff'] = df_compare['MWh'] - df_compare['MWh_year']
df_compare[df_compare['diff'] > 0.1].head(100)

,,MWh,MWh_year,diff
country,technology,,,
AT,RunOfRiver,2.870686e+07,2.862780e+07,79063.2000
BE,RunOfRiver,2.622973e+05,2.615921e+05,705.1900
BG,RunOfRiver,9.749908e+05,9.718508e+05,3140.0000
CH,RunOfRiver,1.598663e+07,1.598404e+07,2588.7900
CZ,RunOfRiver,4.430842e+05,4.410963e+05,1987.9000
DE,RunOfRiver,1.595960e+07,1.591874e+07,40863.8175
ES,RunOfRiver,5.281766e+06,5.257354e+06,24412.0000
FR,RunOfRiver,2.867511e+07,2.854835e+07,126752.0000
GB,RunOfRiver,6.043630e+06,6.033579e+06,10051.5000


check for missing countries in df_annual

In [57]:
missing = np.setdiff1d(countries,np.unique(df_annual.reset_index().country.unique()))
missing

array([], dtype='<U2')

## Load

Get load and select countries and baseyear.

In [58]:
df_load_in_entsoe = pd.read_csv(fn_load_entsoe, parse_dates=True, index_col="date")[countries].copy()
df_load_in_eurostat = pd.read_csv(fn_load_eurostat, index_col="country")[str(baseyear)].copy()
df_load_in_eurostat = pd.DataFrame(df_load_in_eurostat[df_load_in_eurostat.index.isin(countries)])
#we have some missing values which we fill with value from previous hour (up to 10 hours)
df_load_in_entsoe = df_load_in_entsoe.fillna(method = 'pad', limit=10) 
df_load_in_entsoe.info() #looks ok apart from LT - not too important for now...

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 8760 entries, 2017-01-01 00:00:00 to 2017-12-31 23:00:00
Data columns (total 25 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AT      8760 non-null   float64
 1   BE      8760 non-null   float64
 2   BG      8733 non-null   float64
 3   HR      8760 non-null   float64
 4   CZ      8760 non-null   float64
 5   DK      8760 non-null   float64
 6   FI      8760 non-null   float64
 7   FR      8757 non-null   float64
 8   DE      8760 non-null   float64
 9   GR      8760 non-null   float64
 10  HU      8760 non-null   float64
 11  IE      8758 non-null   float64
 12  IT      8760 non-null   float64
 13  LU      8760 non-null   float64
 14  NL      8760 non-null   float64
 15  PL      8760 non-null   float64
 16  PT      8760 non-null   float64
 17  RO      8760 non-null   float64
 18  SK      8760 non-null   float64
 19  SI      8760 non-null   float64
 20  ES      8760 non-null   float64
 21  S

In [59]:
#we fill the rest with 0s for now
df_load_in_entsoe = df_load_in_entsoe.fillna(0)
df_load_in_entsoe.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 8760 entries, 2017-01-01 00:00:00 to 2017-12-31 23:00:00
Data columns (total 25 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AT      8760 non-null   float64
 1   BE      8760 non-null   float64
 2   BG      8760 non-null   float64
 3   HR      8760 non-null   float64
 4   CZ      8760 non-null   float64
 5   DK      8760 non-null   float64
 6   FI      8760 non-null   float64
 7   FR      8760 non-null   float64
 8   DE      8760 non-null   float64
 9   GR      8760 non-null   float64
 10  HU      8760 non-null   float64
 11  IE      8760 non-null   float64
 12  IT      8760 non-null   float64
 13  LU      8760 non-null   float64
 14  NL      8760 non-null   float64
 15  PL      8760 non-null   float64
 16  PT      8760 non-null   float64
 17  RO      8760 non-null   float64
 18  SK      8760 non-null   float64
 19  SI      8760 non-null   float64
 20  ES      8760 non-null   float64
 21  S

In [60]:
df_load_in_entsoe_year = df_load_in_entsoe.resample('Y').sum().T.reset_index()
df_load_in_entsoe_year.columns = ['country',baseyear]
df_load_in_entsoe_year = df_load_in_entsoe_year.set_index('country')

calculate and apply weighting based on eurostat yearly values

In [61]:
df_load_weighting = df_load_in_eurostat[str(baseyear)]/df_load_in_entsoe_year[baseyear]
df_load_weighting[df_load_weighting.isna()] = 1 #Switzerland is missing from Eurostat data, so we assign a value of 1
df_load = df_load_in_entsoe * df_load_weighting.T
df_load = pd.DataFrame(df_load.stack()).reset_index().rename(columns=
                                                            {'date':'time','level_1':'country',0:'MWh'}).set_index(['country','time'])
df_load.head()

,,MWh
country,time,
AT,2017-01-01,7742.689434
BE,2017-01-01,10404.221054
BG,2017-01-01,4662.164944
CH,2017-01-01,6536.400000
CZ,2017-01-01,6655.923967


correct load by trade with countries outside of the system

In [62]:
df_trade = pd.read_csv(fn_trade, parse_dates=True, index_col="time")
df_trade.head()

,from_country,to_country,MWh
time,,,
2017-01-01 00:00:00,AL,GR,230.0
2017-01-01 01:00:00,AL,GR,250.0
2017-01-01 02:00:00,AL,GR,250.0
2017-01-01 03:00:00,AL,GR,250.0
2017-01-01 04:00:00,AL,GR,250.0


In [63]:
df_trade_import = df_trade[(df_trade.from_country.isin(countries_trade) & df_trade.to_country.isin(countries))]
df_trade_import = df_trade_import.groupby(['time','to_country']).sum()
df_trade_import = df_trade_import.reset_index().rename(columns={'to_country':'country','MWh':'import'}).set_index(['country','time'])
df_trade_import.head()

,,import
country,time,
BG,2017-01-01,86.00
FI,2017-01-01,80.50
GB,2017-01-01,824.62
GR,2017-01-01,620.00
HR,2017-01-01,20.00


In [64]:
df_trade_export = df_trade[(df_trade.from_country.isin(countries) & df_trade.to_country.isin(countries_trade))]
df_trade_export = df_trade_export.groupby(['time','from_country']).sum()
df_trade_export = df_trade_export.reset_index().rename(columns={'from_country':'country','MWh':'export'}).set_index(['country','time'])
df_trade_export.head()

,,export
country,time,
BG,2017-01-01,379.0
FI,2017-01-01,0.0
GB,2017-01-01,295.5
GR,2017-01-01,0.0
HR,2017-01-01,773.0


In [65]:
df_load_corrected = df_load.merge(df_trade_export,how='left',left_index=True,right_index=True)
df_load_corrected = df_load_corrected.merge(df_trade_import,how='left',left_index=True,right_index=True)
df_load_corrected = df_load_corrected.fillna(0)
df_load_corrected['MWh_corrected'] = df_load_corrected['MWh'] + df_load_corrected['export'] - df_load_corrected['import']
df_load_corrected.head()

,,MWh,export,import,MWh_corrected
country,time,,,,
AT,2017-01-01,7742.689434,0.0,0.0,7742.689434
BE,2017-01-01,10404.221054,0.0,0.0,10404.221054
BG,2017-01-01,4662.164944,379.0,86.0,4955.164944
CH,2017-01-01,6536.400000,0.0,0.0,6536.400000
CZ,2017-01-01,6655.923967,0.0,0.0,6655.923967


In [66]:
df_load = pd.DataFrame()
df_load = df_load_corrected[['MWh_corrected']].copy().rename(columns={'MWh_corrected':'MWh'})
df_load.head()

,,MWh
country,time,
AT,2017-01-01,7742.689434
BE,2017-01-01,10404.221054
BG,2017-01-01,4955.164944
CH,2017-01-01,6536.400000
CZ,2017-01-01,6655.923967


### Rescale demand to fit system generation

In [67]:
#load is higher than generation so we need some scaling in the model
load_scaling = df_generation['MWh'].sum()/df_load['MWh'].sum()
df_load['MWh'] = df_load['MWh'] * load_scaling
df_generation['MWh'].sum()/df_load['MWh'].sum()

1.0

In [68]:
load_scaling

0.9607889823844155

## Reservoir inflows

In [69]:
df_inflow_in = pd.read_csv(fn_reservoir_inflow)
df_inflow_in = df_inflow_in[df_inflow_in.country.isin(countries)].copy()
df_inflow_in = df_inflow_in[df_inflow_in.year == np.int64(baseyear)].copy()
df_inflow_in = df_inflow_in.set_index(['country','year','week'])
df_inflow_in.head(1)

,,,Cumulated inflow into reservoirs per week in GWh,Cumulated NATURAL inflow into the pump-storage reservoirs per week in GWh
country,year,week,,
AT,2017,1,7.877815,23.988208


In [70]:
#now we distribute values to hours
df_inflow_reservoirs = pd.DataFrame(df_inflow_in['Cumulated inflow into reservoirs per week in GWh']).reset_index()
#convert to MWh per hour
df_inflow_reservoirs['MWh'] = df_inflow_reservoirs['Cumulated inflow into reservoirs per week in GWh'] * 1000 / (7*24)
df_inflow_reservoirs = df_inflow_reservoirs[['year','week','country','MWh']]
#assign timestamps
df_inflow_reservoirs['week'] = (df_inflow_reservoirs['week']).apply(lambda x: '{0:0>2}'.format(x))
df_inflow_reservoirs['date'] = df_inflow_reservoirs['year'].astype(str) + '-W' + df_inflow_reservoirs['week'].astype(str) +'-1'
df_inflow_reservoirs['date'] = pd.to_datetime((df_inflow_reservoirs['date']), format='%Y-W%W-%w')
df_inflow_reservoirs =  pd.DataFrame(df_inflow_reservoirs[['date','country','MWh']])
#add timestamp for first hour for all countries for upsamling in next step
for country in countries:
    df_inflow_reservoirs = df_inflow_reservoirs.append(pd.DataFrame([[pd.Timestamp(str(np.int64(baseyear)-1)+'-12-31'),country,float("NaN")]], columns=['date','country','MWh']), ignore_index=True)
#then upsample with backwardsfill and select baseyear again
df_inflow_reservoirs.head(1)

,date,country,MWh
0,2017-01-02,AT,46.891756


In [71]:
#upsample to all hours
df_inflow_reservoirs_hourly = df_inflow_reservoirs.pivot(index='date',columns='country')[['MWh']].resample('H').fillna("bfill").stack()
df_inflow_reservoirs_hourly = df_inflow_reservoirs_hourly[pd.DatetimeIndex(df_inflow_reservoirs_hourly.reset_index().date).year == baseyear]
df_inflow_reservoirs_hourly.head(1)

,,MWh
date,country,
2017-01-01,AT,46.891756


check for missing countries

In [72]:
missing = np.setdiff1d(countries,np.unique(df_inflow_reservoirs_hourly.reset_index().country.unique()))
missing
#look ok

array(['DK', 'FI', 'NL'], dtype='<U2')

Do the same steps for pump storage reservoirs

In [73]:
#now we distribute values to hours
df_inflow_pump_reservoirs = pd.DataFrame(df_inflow_in['Cumulated NATURAL inflow into the pump-storage reservoirs per week in GWh']).reset_index()
#convert to MWh per hour
df_inflow_pump_reservoirs['MWh'] = df_inflow_pump_reservoirs['Cumulated NATURAL inflow into the pump-storage reservoirs per week in GWh'] * 1000 / (7*24)
df_inflow_pump_reservoirs = df_inflow_pump_reservoirs[['year','week','country','MWh']]
#assign timestamps
df_inflow_pump_reservoirs['week'] = (df_inflow_pump_reservoirs['week']).apply(lambda x: '{0:0>2}'.format(x))
df_inflow_pump_reservoirs['date'] = df_inflow_pump_reservoirs['year'].astype(str) + '-W' + df_inflow_pump_reservoirs['week'].astype(str) +'-1'
df_inflow_pump_reservoirs['date'] = pd.to_datetime((df_inflow_pump_reservoirs['date']), format='%Y-W%W-%w')
df_inflow_pump_reservoirs =  pd.DataFrame(df_inflow_pump_reservoirs[['date','country','MWh']])
df_inflow_pump_reservoirs.head(1)

,date,country,MWh
0,2017-01-02,AT,142.786952


In [74]:
#upsample to all hours
#first, add timestamp for first hour for all countries
for country in countries:
    df_inflow_pump_reservoirs = df_inflow_pump_reservoirs.append(pd.DataFrame([[pd.Timestamp(str(baseyear-1)+'-12-31'),country,float("NaN")]], columns=['date','country','MWh']), ignore_index=True)
#then upsample with backwardsfill and select baseyear again
df_inflow_pump_reservoirs_hourly = df_inflow_pump_reservoirs.pivot(index='date',columns='country')[['MWh']].resample('H').fillna("bfill").stack()
df_inflow_pump_reservoirs_hourly = df_inflow_pump_reservoirs_hourly[pd.DatetimeIndex(df_inflow_pump_reservoirs_hourly.reset_index().date).year == baseyear]
df_inflow_pump_reservoirs_hourly.head()

MWh
date       country            
2017-01-01 AT       142.786952
           BE         0.000000
           BG        45.510750
           CH         0.000000
           CZ         0.000000

check for missing countries

In [75]:
missing = np.setdiff1d(countries,np.unique(df_inflow_pump_reservoirs_hourly.reset_index().country.unique()))
missing
#looks ok

array(['DK', 'FI', 'NL'], dtype='<U2')

In [76]:
#Now we add a column for technology for both dfs and merge them
df_inflow_reservoirs_hourly['technology'] = 'Reservoir'
df_inflow_pump_reservoirs_hourly['technology'] = 'PumpOpen'
df_storage_inflows = df_inflow_reservoirs_hourly.append(df_inflow_pump_reservoirs_hourly).reset_index()
#lastly, we delete 0 values
df_storage_inflows = df_storage_inflows[df_storage_inflows['MWh'] != 0]
df_storage_inflows.head()

,date,country,MWh,technology
0,2017-01-01,AT,46.891756,Reservoir
2,2017-01-01,BG,232.966743,Reservoir
3,2017-01-01,CH,732.387192,Reservoir
4,2017-01-01,CZ,37.869048,Reservoir
5,2017-01-01,DE,10.009650,Reservoir


Unfortunately, values for Norway are missing for 2017, so we derive them from ENTSO-E Transparency generation and inflow and distribute them according to per technology share

In [77]:
#we load weekly values as this is best temporal resolution for storage levels
df_reservoir_level_weekly = pd.read_csv(fn_reservoir_profile_weekly_TP,parse_dates=True, index_col="date").reset_index()
df_gen_weekly_in =  pd.read_csv(fn_gen_entsoe_weekly,parse_dates=True, index_col="date").reset_index()

In [78]:
#only keep reservoirs and net_generation, pumping is included here
df_gen_weekly_reservoir = df_gen_weekly_in[df_gen_weekly_in.tech=='Reservoir'][['date','country','net_generation']]
#adjust datetime for gen_weekly as reservoir level is one day later in 2017
if baseyear == 2017:
    df_gen_weekly_reservoir.date = df_gen_weekly_reservoir.date + datetime.timedelta(days=1)
df_gen_weekly_reservoir = df_gen_weekly_reservoir.set_index(['date','country'])
df_gen_weekly_reservoir.head()

net_generation
date       country                
2017-01-02 AT             7806.000
           BG             3053.000
           CH            20395.850
           CZ              575.780
           DE             1004.045

In [79]:
#create a df with reservoir levels from previus week
df_reservoir_level_weekly_before = df_reservoir_level_weekly.copy()
df_reservoir_level_weekly_before.date = df_reservoir_level_weekly_before.date + datetime.timedelta(weeks=1)
df_reservoir_level_weekly_before = df_reservoir_level_weekly_before.rename(columns = {'MWh':'MWh_t-1'})
df_reservoir_level_weekly_before.head()

,date,country,MWh_t-1
0,2017-01-09,LV,8707.0
1,2017-01-09,HR,551330.0
2,2017-01-09,IT,2863629.0
3,2017-01-09,FI,3415000.0
4,2017-01-09,BG,1117088.0


In [80]:
#merge all three dfs and calculate inflows
df_reservoir_level_weekly_merge = df_reservoir_level_weekly.merge(df_reservoir_level_weekly_before,
                                                            how='left',on=['date','country']).dropna()
df_reservoir_level_weekly_merge[df_reservoir_level_weekly_merge.country == 'NO'].head()
df_reservoir_level_weekly_merged = df_reservoir_level_weekly_merge.merge(df_gen_weekly_reservoir,
                                                                        how='left',on=['date','country']).dropna()
df_reservoir_level_weekly_merged['natural_inflow'] = df_reservoir_level_weekly_merged['MWh'] - df_reservoir_level_weekly_merged['MWh_t-1'] + df_reservoir_level_weekly_merged['net_generation']
df_reservoir_level_weekly_merged[df_reservoir_level_weekly_merged.country == 'NO'].head()

,date,country,MWh,MWh_t-1,net_generation,natural_inflow
12,2017-01-09,NO,50102000.0,51959000.0,3339635.79,1482635.79
25,2017-01-16,NO,48484000.0,50102000.0,2977521.64,1359521.64
48,2017-01-23,NO,46780000.0,48484000.0,3126057.99,1422057.99
62,2017-01-30,NO,44240000.0,46780000.0,3102274.73,562274.73
70,2017-02-06,NO,41458000.0,44240000.0,3274730.61,492730.61


In [81]:
#eventually, upsample inflows to hourly values
df_reservoir_inflow_entsoe = df_reservoir_level_weekly_merged.copy()[['date','country','natural_inflow']]
df_reservoir_inflow_entsoe['natural_inflow'][df_reservoir_inflow_entsoe['natural_inflow'] < 0] = 0
df_reservoir_inflow_entsoe['natural_inflow'] = df_reservoir_inflow_entsoe['natural_inflow'] / (7*24)
#first, add timestamp for first hour for all countries
for country in countries:
    df_reservoir_inflow_entsoe = df_reservoir_inflow_entsoe.append(pd.DataFrame([[pd.Timestamp(str(baseyear-1)+'-12-31'),country,float("NaN")]], columns=['date','country','natural_inflow']), ignore_index=True)
    df_reservoir_inflow_entsoe = df_reservoir_inflow_entsoe.append(pd.DataFrame([[pd.Timestamp(str(baseyear+1)+'-01-01'),country,float("NaN")]], columns=['date','country','natural_inflow']), ignore_index=True)
#then upsample with backwardsfill and select baseyear again
df_reservoir_inflow_entsoe_hourly = df_reservoir_inflow_entsoe.pivot(index='date',columns='country')[['natural_inflow']].resample('H').fillna("pad")
df_reservoir_inflow_entsoe_hourly = df_reservoir_inflow_entsoe_hourly[['natural_inflow']].fillna(method="bfill").stack()
df_reservoir_inflow_entsoe_hourly = df_reservoir_inflow_entsoe_hourly[pd.DatetimeIndex(df_reservoir_inflow_entsoe_hourly.reset_index().date).year == baseyear].reset_index()
df_reservoir_inflow_entsoe_hourly.head()

,date,country,natural_inflow
0,2017-01-01,AT,87.213512
1,2017-01-01,BG,0.000000
2,2017-01-01,CH,0.000000
3,2017-01-01,ES,955.215476
4,2017-01-01,FR,1689.744048


In [82]:
#now we fill missing countries from before (turns out to be only NO - will be empty in other years)
missing_inflows = np.setdiff1d(countries,np.unique(df_storage_inflows.reset_index().country.unique()))
df_reservoir_inflow_entsoe_filler = df_reservoir_inflow_entsoe_hourly[df_reservoir_inflow_entsoe_hourly['country'].isin(missing_inflows)].copy()
df_reservoir_inflow_entsoe_filler.country.unique()

array(['NO'], dtype=object)

In [83]:
#Norway has only PumpOpen
df_reservoir_inflow_entsoe_filler['technology'] = 'PumpOpen'
df_reservoir_inflow_entsoe_filler = df_reservoir_inflow_entsoe_filler.rename(columns={'natural_inflow':'MWh'})
df_reservoir_inflow_entsoe_filler.head()

,date,country,MWh,technology
7,2017-01-01 00:00:00,NO,8825.213036,PumpOpen
19,2017-01-01 01:00:00,NO,8825.213036,PumpOpen
31,2017-01-01 02:00:00,NO,8825.213036,PumpOpen
43,2017-01-01 03:00:00,NO,8825.213036,PumpOpen
55,2017-01-01 04:00:00,NO,8825.213036,PumpOpen


In [84]:
#and add this to storage inflow df
df_storage_inflows = df_storage_inflows.append(df_reservoir_inflow_entsoe_filler)

## Reservoir levels
Get hourly reservoir levels from weekly levels - to be used as storage start and end conditions

first, we load reservoir size to be able to distribute value by technology

In [85]:
df_reservoir = pd.read_csv(fn_reservoir_size)
df_reservoir = df_reservoir.rename(columns={
    'Pump Storage - Closed Loop - Cumulated (upper or head) reservoir capacity (GWh)':'PumpClosed',
    'Pump Storage - Open Loop - Cumulated (upper or head) reservoir capacity (GWh)':'PumpOpen',
    'Reservoir - Reservoir capacity (GWh)':'Reservoir',
    'Run-of-River and pondage - Reservoir capacity linked to Run of River and Pondage units (GWh)':'RunOfRiver'
                                }).set_index('country')
df_reservoir = df_reservoir[['PumpClosed','PumpOpen','Reservoir','RunOfRiver']]
#temporary delete run of river from next equation: + df_reservoir['RunOfRiver']
df_reservoir = df_reservoir.drop(columns={'RunOfRiver'})
df_reservoir['sum'] = df_reservoir['PumpClosed'] + df_reservoir['PumpOpen'] + df_reservoir['Reservoir']
df_reservoir = pd.DataFrame(df_reservoir.stack()).reset_index()
df_reservoir.columns = ['country','technology','capacity_GWh']
df_reservoir['capacity_MWh'] = df_reservoir['capacity_GWh']*1000
df_reservoir = df_reservoir.pivot(index = 'country', columns = 'technology', values='capacity_MWh')
df_reservoir.head(1)
#qgrid.show_grid(df_reservoir.set_index(['country','technology'])['capacity_GWh'], show_toolbar=True)

technology,PumpClosed,PumpOpen,Reservoir,sum
country,,,,
AL,0.0,0.0,1450000.0,1450000.0


replace absolute values with shares

In [86]:
df_reservoir_shares = df_reservoir.copy()
df_reservoir_shares['PumpClosed']  = df_reservoir_shares['PumpClosed'] / df_reservoir_shares['sum']
df_reservoir_shares['PumpOpen'] = df_reservoir_shares['PumpOpen'] / df_reservoir_shares['sum']
df_reservoir_shares['Reservoir'] = df_reservoir_shares['Reservoir'] / df_reservoir_shares['sum']
#df_reservoir_shares['RunOfRiver'] = df_reservoir_shares['RunOfRiver'] / df_reservoir_shares['sum']
df_reservoir_shares = df_reservoir_shares[['PumpClosed','PumpOpen','Reservoir']]
df_reservoir_shares.head(1)

technology,PumpClosed,PumpOpen,Reservoir
country,,,
AL,0.0,0.0,1.0


now, we load values from ENTSO-E Transparency platform (other dataset has only 'AL', 'AT', 'CH', 'ES', 'FI', 'FR', 'PT', 'SE')

In [87]:
df_reservoir_level = pd.read_csv(fn_reservoir_profile_TP, parse_dates=True, index_col='date')[str(baseyear)].reset_index().set_index(['date','country'])
df_reservoir_level.head(1)

C:\Users\JSAVEL~1\AppData\Local\Temp/ipykernel_11892/4193697556.py:1: FutureWarning: Indexing a DataFrame with a datetimelike index using a single string to slice the rows, like `frame[string]`, is deprecated and will be removed in a future version. Use `frame.loc[string]` instead.
  df_reservoir_level = pd.read_csv(fn_reservoir_profile_TP, parse_dates=True, index_col='date')[str(baseyear)].reset_index().set_index(['date','country'])


,,MWh
date,country,
2017-01-01,AT,825977.57


and merge them with the storage sizes per technology

In [88]:
df_reservoir_level_merged = df_reservoir_level.reset_index().merge(df_reservoir_shares.reset_index(),
                                                                  how='left',left_on='country',
                                                                  right_on='country')
df_reservoir_level_merged['PumpClosed']  = df_reservoir_level_merged['PumpClosed'] * df_reservoir_level_merged['MWh']
df_reservoir_level_merged['PumpOpen'] = df_reservoir_level_merged['PumpOpen'] * df_reservoir_level_merged['MWh']
df_reservoir_level_merged['Reservoir'] = df_reservoir_level_merged['Reservoir'] * df_reservoir_level_merged['MWh']
#df_reservoir_level_merged['RunOfRiver'] = df_reservoir_level_merged['RunOfRiver'] * df_reservoir_level_merged['MWh']
df_reservoir_level_merged = pd.DataFrame(df_reservoir_level_merged.set_index(['date','country'])[
    ['PumpClosed','PumpOpen','Reservoir',]].stack()).reset_index().rename(
    columns={'level_2':'technology',0:'MWh'}).set_index(['date','country','technology'])
df_reservoir_level_merged.head(1)

,,,MWh
date,country,technology,
2017-01-01,AT,PumpClosed,0.0


In [89]:
#eventually, we create a df that has both relative level in % and absolute value in MWh

In [90]:
df_reservoir_level_final = df_reservoir_level_merged.reset_index().merge(
    pd.DataFrame(df_reservoir[['PumpClosed','PumpOpen','Reservoir']].stack()).rename(columns={0:'size_MWh'}).reset_index(),
    how='outer',on=['country','technology'])
df_reservoir_level_final['level'] = df_reservoir_level_final['MWh'] / df_reservoir_level_final['size_MWh']
df_reservoir_level_final = df_reservoir_level_final.fillna(0)
df_reservoir_level_final = df_reservoir_level_final[df_reservoir_level_final.country.isin(countries)]
df_reservoir_level_final.head()

,date,country,technology,MWh,size_MWh,level
0,2017-01-01 00:00:00,AT,PumpClosed,0.0,0.0,0.0
1,2017-01-01 01:00:00,AT,PumpClosed,0.0,0.0,0.0
2,2017-01-01 02:00:00,AT,PumpClosed,0.0,0.0,0.0
3,2017-01-01 03:00:00,AT,PumpClosed,0.0,0.0,0.0
4,2017-01-01 04:00:00,AT,PumpClosed,0.0,0.0,0.0


In [91]:
#check which countries do not have values assigned
missing = np.setdiff1d(countries,np.unique(df_reservoir_level_final.reset_index().country.unique()))
missing
#this looks ok

array(['DK', 'NL'], dtype='<U2')

## Reservoir size

we use the previous df, but in some cases, level is greater than size so we increase size to max level

In [92]:
df_reservoir_size = df_reservoir_level_final.drop(columns={'date'}).groupby(['country','technology']).max()
df_reservoir_size['size_MWh'][df_reservoir_size['MWh'] > df_reservoir_size['size_MWh']] = df_reservoir_size['MWh']
df_reservoir_size = pd.DataFrame(df_reservoir_size['size_MWh']).rename(columns={'size_MWh':'MWh'})
df_reservoir_size.head(1)

,,MWh
country,technology,
AT,PumpClosed,0.0


## Net transfer capacity

Get net transfer capacities and select countries

In [93]:
df_ntc_in = pd.read_csv(fn_ntc, parse_dates=True, index_col="date")
df_ntc  = df_ntc_in[(df_ntc_in["from"].isin(countries))
                     & (df_ntc_in["to"].isin(countries))][str(baseyear)].reset_index()
df_ntc.info()
df_ntc.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 657000 entries, 0 to 656999
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype         
---  ------  --------------   -----         
 0   date    657000 non-null  datetime64[ns]
 1   from    657000 non-null  object        
 2   to      657000 non-null  object        
 3   ntc     657000 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(2)
memory usage: 20.1+ MB


C:\Users\JSAVEL~1\AppData\Local\Temp/ipykernel_11892/1900437732.py:2: FutureWarning: Indexing a DataFrame with a datetimelike index using a single string to slice the rows, like `frame[string]`, is deprecated and will be removed in a future version. Use `frame.loc[string]` instead.
  df_ntc  = df_ntc_in[(df_ntc_in["from"].isin(countries))


,date,from,to,ntc
0,2017-01-01,AT,CH,1200.0
1,2017-01-01,AT,CZ,900.0
2,2017-01-01,AT,DE,5000.0
3,2017-01-01,AT,IT,405.0
4,2017-01-01,BE,FR,1800.0


Check if all countries have NTC values assigned

In [94]:
missing = np.setdiff1d(countries,np.unique(df_ntc['from']))
print(missing)

[]


## Hourly day ahead trade data

In [95]:
df_trade = pd.read_csv(fn_trade, parse_dates=True, index_col="time")
df_trade  = df_trade[(df_trade["from_country"].isin(countries))
                     & (df_trade["to_country"].isin(countries))]
df_trade.head()

,from_country,to_country,MWh
time,,,
2017-01-01 00:00:00,AT,CH,850.0
2017-01-01 01:00:00,AT,CH,850.0
2017-01-01 02:00:00,AT,CH,850.0
2017-01-01 03:00:00,AT,CH,850.0
2017-01-01 04:00:00,AT,CH,850.0


In [96]:
missing = np.setdiff1d(countries,np.unique(df_trade['from_country']))
print(missing)

[]


## Prices

Get prices and select countries and baseyear. 

In [97]:
df_price_in = pd.read_csv(fn_price, parse_dates=True, index_col="date")[str(baseyear)][countries].copy()
df_price_in.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 8760 entries, 2017-01-01 00:00:00 to 2017-12-31 23:00:00
Data columns (total 25 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AT      8760 non-null   float64
 1   BE      8760 non-null   float64
 2   BG      8760 non-null   float64
 3   HR      1802 non-null   float64
 4   CZ      8760 non-null   float64
 5   DK      8760 non-null   float64
 6   FI      8760 non-null   float64
 7   FR      8760 non-null   float64
 8   DE      8760 non-null   float64
 9   GR      8760 non-null   float64
 10  HU      8760 non-null   float64
 11  IE      8736 non-null   float64
 12  IT      8760 non-null   float64
 13  LU      8760 non-null   float64
 14  NL      8760 non-null   float64
 15  PL      8566 non-null   float64
 16  PT      8760 non-null   float64
 17  RO      8736 non-null   float64
 18  SK      8760 non-null   float64
 19  SI      8760 non-null   float64
 20  ES      8760 non-null   float64
 21  S

C:\Users\JSAVEL~1\AppData\Local\Temp/ipykernel_11892/419932068.py:1: FutureWarning: Indexing a DataFrame with a datetimelike index using a single string to slice the rows, like `frame[string]`, is deprecated and will be removed in a future version. Use `frame.loc[string]` instead.
  df_price_in = pd.read_csv(fn_price, parse_dates=True, index_col="date")[str(baseyear)][countries].copy()


** NOTE: Here be careful about missing values. Correction should be done by hand.**

So we have a missing day for Switzerland, one for Ireland, and four hours for Poland. We assign values from Germany to Switzerland and Poland. And GB values to Ireland: 

In [98]:
df_price = df_price_in.copy()
df_price.CH = df_price.CH.fillna(df_price_in.DE)
df_price.PL = df_price.PL.fillna(df_price_in.DE)
df_price.IE = df_price.IE.fillna(df_price_in.GB)
df_price.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 8760 entries, 2017-01-01 00:00:00 to 2017-12-31 23:00:00
Data columns (total 25 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AT      8760 non-null   float64
 1   BE      8760 non-null   float64
 2   BG      8760 non-null   float64
 3   HR      1802 non-null   float64
 4   CZ      8760 non-null   float64
 5   DK      8760 non-null   float64
 6   FI      8760 non-null   float64
 7   FR      8760 non-null   float64
 8   DE      8760 non-null   float64
 9   GR      8760 non-null   float64
 10  HU      8760 non-null   float64
 11  IE      8760 non-null   float64
 12  IT      8760 non-null   float64
 13  LU      8760 non-null   float64
 14  NL      8760 non-null   float64
 15  PL      8760 non-null   float64
 16  PT      8760 non-null   float64
 17  RO      8736 non-null   float64
 18  SK      8760 non-null   float64
 19  SI      8760 non-null   float64
 20  ES      8760 non-null   float64
 21  S

Furthermore, if we add EU-27 countries, HR and RO are missing values
    using SI for HR (seems to be the neighbour with most similar prices)
    and BG for RO
    (used this as rough proxy: https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:52019DC0001&from=EN)

In [99]:
df_price = df_price.copy()
df_price.HR = df_price.HR.fillna(df_price_in.SI)
df_price.RO = df_price.RO.fillna(df_price_in.BG)
df_price.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 8760 entries, 2017-01-01 00:00:00 to 2017-12-31 23:00:00
Data columns (total 25 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AT      8760 non-null   float64
 1   BE      8760 non-null   float64
 2   BG      8760 non-null   float64
 3   HR      8760 non-null   float64
 4   CZ      8760 non-null   float64
 5   DK      8760 non-null   float64
 6   FI      8760 non-null   float64
 7   FR      8760 non-null   float64
 8   DE      8760 non-null   float64
 9   GR      8760 non-null   float64
 10  HU      8760 non-null   float64
 11  IE      8760 non-null   float64
 12  IT      8760 non-null   float64
 13  LU      8760 non-null   float64
 14  NL      8760 non-null   float64
 15  PL      8760 non-null   float64
 16  PT      8760 non-null   float64
 17  RO      8760 non-null   float64
 18  SK      8760 non-null   float64
 19  SI      8760 non-null   float64
 20  ES      8760 non-null   float64
 21  S

## Cost Resource Curves

In [100]:
df_cost_res_in = pd.read_csv(fn_cost_res)

In [101]:
df_cost_res = df_cost_res_in[(df_cost_res_in.country.isin(countries))
                  & df_cost_res_in.technology.isin(technologies)].copy()

Add plant index to dataframe

In [102]:
df_cost_res["plant"] = df_cost_res.country + "_" + df_cost_res.technology
df_cost_res.head(1)

,country,technology,cinv_0,cinv_1,potential_twh,potential_mw,plant
0,AT,Solar,36.213953,2.073156e-08,86.530179,79833.267539,AT_Solar


check for missing countries in df_cost_res

In [103]:
missing = np.setdiff1d(countries,np.unique(df_cost_res.country.unique()))
missing

array([], dtype='<U2')

## CHP Data

In [104]:
df_chp_gen_in = pd.read_csv(fn_chp_gen)
df_chp_gen_in.head(1)

,year,country,tech,MWh
0,2015,BE,Biomass,929388.799836


Select year and countries:

In [105]:
df_chp_gen = df_chp_gen_in[(df_chp_gen_in.year == baseyear)
                          & (df_chp_gen_in.country.isin(countries))].drop("year", axis = 1).rename(
columns={'tech':'technology'})
df_chp_gen.head()

,country,technology,MWh
58,BE,Biomass,1.079709e+06
59,BG,Biomass,4.569863e+04
60,CZ,Biomass,3.366929e+05
61,DK,Biomass,3.682033e+06
62,DE,Biomass,1.227804e+07


check for missing countries in df_chp_gen

In [106]:
missing = np.setdiff1d(countries,np.unique(df_chp_gen.country.unique()))
missing
#only CH missing from eurostat data - CHP not relevant for Nuclear since baseload

array(['CH'], dtype='<U2')

Also get profiles and add a date for the current baseyear

In [107]:
df_heat_dem_in = pd.read_csv(fn_heat_dem)
dates = pd.date_range(start='%d-01-01 00:00:00' % baseyear, 
                      end='%d-12-31 23:00:00' % baseyear, 
                      periods = len(df_heat_dem_in))
df_heat_dem_in = df_heat_dem_in.set_index(dates)
df_heat_dem_in.index.name = "date"
df_heat_dem_in = df_heat_dem_in.reset_index()
df_heat_dem_in.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   date                  8760 non-null   datetime64[ns]
 1   heat_demand           8760 non-null   float64       
 2   heat_demand_relative  8760 non-null   float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 205.4 KB


# Upward adjustment of capacities if hourly entsoe value is higher

In [108]:
df_hourly_entsoe_max = df_hourly_entsoe.reset_index().groupby(['country','technology']).max()
df_hourly_entsoe_max = df_hourly_entsoe_max.reset_index().drop(columns={'time'})
df_hourly_entsoe_max = df_hourly_entsoe_max[(df_hourly_entsoe_max.country.isin(countries))
                  & df_hourly_entsoe_max.technology.isin(technologies)]
df_hourly_entsoe_max.head()

,country,technology,MWh_hourly
0,AT,Biomass,328.00
1,AT,Gas,4214.20
2,AT,HardCoal,537.10
3,AT,Oil,0.00
4,AT,Other,122.07


In [112]:
df_cap_test = df_cap.copy().reset_index()
df_cap_test = df_cap_test.merge(df_hourly_entsoe_max,left_on=['country','technology'],right_on=['country','technology'],how='left')
df_cap_test.head()

,technology,country,capacity,pumping,MWh_hourly
0,Biomass,AT,572.0,0.0,328.00
1,Gas,AT,4841.0,0.0,4214.20
2,HardCoal,AT,598.0,0.0,537.10
3,Oil,AT,166.0,0.0,0.00
4,Other,AT,978.0,0.0,122.07


In [114]:
#df_cap_test[df_cap_test.MWh_hourly > df_cap_test.capacity]

In [115]:
df_cap_test.capacity[df_cap_test.MWh_hourly > df_cap_test.capacity] = df_cap_test.MWh_hourly

C:\Users\JSAVEL~1\AppData\Local\Temp/ipykernel_11892/216730869.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cap_test.capacity[df_cap_test.MWh_hourly > df_cap_test.capacity] = df_cap_test.MWh_hourly


In [116]:
df_cap_test = df_cap_test.set_index(['technology','country'])

# GDX export

We create two different datasets: (1) all hours, (2) one day per month

To facilitate injecting data into the gdx under multiple time specifications, a function to inject static data. Note that the function only specifies the gdx file as input. Data parameters are taken from outside, i.e., global name space and, therefore, have to be defined before:

In [117]:
list(df_cost_in[["Technology", "Main Fuel"]].itertuples(index=False, name=None))

[('Nuclear', 'Uran'),
 ('Lignite', 'Lignite'),
 ('Gas', 'Gas'),
 ('Oil', 'Oil'),
 ('RunOfRiver', 'Hydro'),
 ('Reservoir', 'Hydro'),
 ('PumpOpen', 'Hydro'),
 ('PumpClosed', 'Hydro'),
 ('Biomass', 'Biomass'),
 ('Solar', 'None'),
 ('WindOffshore', 'None'),
 ('WindOnshore', 'None'),
 ('HardCoal', 'HardCoal'),
 ('Other', 'Other'),
 ('GasCCS', 'Gas')]

In [118]:
def inject_static(gdx):
    """Injects static data into gdx file
    :param gdx: <gams.GamsDatabase> gdx to inject data
    """
    # overwrite settings (False does not work due to bug in gdxtools)
    overwrite = True
    
    # sets
    ## technology sets
    gt.add_set(sorted(countries), "c", gdx, text="countries", overwrite=overwrite)
    gt.add_set(sorted(countries_EU_27), "eu", gdx, text="countries", overwrite=overwrite)
    gt.add_set(technologies, "tech", gdx, text="technologies", overwrite=overwrite)
    gt.add_set(conventionals, "i", gdx, text="conventional technologies", overwrite=overwrite)
    gt.add_set(storages, "s", gdx, text="storage facilities", overwrite=overwrite)  
    gt.add_set(renewables, "r", gdx, text="new renewable sources", overwrite=overwrite)  
    gt.add_set(old_renewables, "ro", gdx, text="old = conventional renewable sources", overwrite=overwrite)  
    gt.add_set(all_renewables, "ra", gdx, text="all renewable sources", overwrite=overwrite)  
    gt.add_set(baseload, "baseload", gdx, text="baseload technologies excl. RoR", overwrite=overwrite)  
    gt.add_set(fixed_feedin, "fixed", gdx, text="fixed feedin technologies", overwrite=overwrite)  
    gt.add_set(peakload, "peak", gdx, text="peak load technologies", overwrite=overwrite)  

    ## fuels 
    gt.add_set(fuels, "f", gdx, text="fuels", overwrite=overwrite)
    gt.add_set(list(df_cost_in[["Technology", "Main Fuel"]].itertuples(index=False, name=None)),
               "mapTF", gdx, text="mapping technology to fuel", overwrite=overwrite)
     
    ## reserve and profiles
    gt.add_set(fuel_price_maps, "map_fuel_price_as", gdx, text="fuel price in c as in cc for fuel f",
               overwrite=overwrite)
    gt.add_set(om_maps, "map_om_cost_as", gdx, text="variable O&M costs in c as in cc for technology tech",
               overwrite=overwrite)

    # country parameters: curtailment penality, elasticities, ...
    gt.add_parameter(df_countries.UpperPriceLimit.to_dict(), "penalty", gdx,
                     text="price cap on hourly electricity prices [Euro per MWh]", overwrite=overwrite)
    gt.add_parameter(df_countries.CurtailmentPenalty.to_dict(), "curtPenalty", gdx,
                     text="curtPenalty [Euro per MWh]", overwrite=overwrite)
    gt.add_parameter(df_countries.ReserveRequirement.to_dict(), "reserveRequirement", gdx,
                     text="reserve requirement [% of hourly demand]", overwrite=overwrite)
    gt.add_parameter(df_countries.DemandElasticity.to_dict(), "epsilon", gdx, 
                  text="price elasticity of demand", overwrite=overwrite)    
    gt.add_parameter(df_countries.CarbonPrice.to_dict(), "p_carb", gdx, 
                  text="carbon price [€ per tonne CO2]", overwrite=overwrite)  
        
    # plant and cost specifications
    df_ = df_cost.set_index(["Technology","country"])
    gt.add_parameter(df_["Average Efficiency"].to_dict(), "eta", gdx, 
                  text="plant efficiency [%]", overwrite=overwrite)    
    gt.add_parameter(df_["Availability"].to_dict(), "availUp", gdx, 
                  text="plant availability [%]", overwrite=overwrite) 
    #gt.add_parameter(df_["variable OM Cost [Euro/MWh]"].to_dict(), "c_vom", gdx, 
    #              text="variable O&M cost [Euro per MWh]", overwrite=overwrite)
    gt.add_parameter(df_["Min Generation"].to_dict(), "min_gen", gdx, 
                  text="minimum generation [% of capacity]", overwrite=overwrite)         
    gt.add_parameter(df_["LoadGradient"].to_dict(), "loadGradient", gdx, 
                  text="variable O&M cost [Euro per MW]", overwrite=overwrite)      
    gt.add_parameter(df_["RampingCost"].to_dict(), "rampingCost", gdx, 
                  text="ramping cost in terms of additional fuel use [MWH per MW]", overwrite=overwrite)     
    gt.add_parameter(df_["RampingCostII"].to_dict(), "rampingCostII", gdx, 
                  text="ramping cost as additional depreciation [Euro per MWh]", overwrite=overwrite)  
    
    # capacities (now also including pumping)
    gt.add_parameter(df_cap_test.capacity.to_dict(), "cap", gdx,    
                  text="installed capacity [MW]", overwrite=overwrite)    
    
    gt.add_parameter(df_cap.pumping.to_dict(), "cap_pump", gdx, 
                  text="installed pumping capacity [MW]", overwrite=overwrite)    
    
    #reservoir size
    df_ = df_reservoir_size.copy()
    gt.add_parameter(df_.MWh.to_dict(), "reservoir_size_up", gdx,
              text="upload reservoir size [MWh]", overwrite=True) 
    
    # cost resource curves for RES investments
    df_cost_res_ = df_cost_res.set_index(["technology", "country"])
    gt.add_parameter(df_cost_res_["cinv_0"].to_dict(), "cinv_0", gdx,
                  text="RES investment cost linear [EUR/MWh]", overwrite=overwrite)
    gt.add_parameter(df_cost_res_["cinv_1"].to_dict(), "cinv_1", gdx,
                  text="RES investment cost quadratic [EUR/MWh^2]", overwrite=overwrite)
    gt.add_parameter((df_cost_res_["potential_twh"]*1000000).to_dict(), "pot_ren_mwh", gdx,
                  text="potential for renewable generation [MWh]", overwrite=overwrite)
    
    
    # fuel specifications
    df_ = df_fuels.set_index("Main Fuel")
    gt.add_parameter(df_["Carbon"].to_dict(), "carb_coef", gdx, 
                  text="carbon coefficent [t per MWh]", overwrite=overwrite)    

    #fuel prices
    df_ = df_fuels_2017.copy()
    gt.add_parameter(df_.set_index(["Main Fuel", "country"]).Price.to_dict(), "pf", gdx,
                  text="fuel price [Euro per MWh]", overwrite=overwrite)
    
    # quadratic part of fuel price
    gt.add_parameter(df_.set_index(["Main Fuel", "country"]).Price_2.to_dict(), "pf_2", gdx, text = "fuel price quadratic part [Euro per MWh^2]",
                  overwrite = overwrite)
    
    # variable O&M cost
    df_ = df_OM_2017.copy()
    gt.add_parameter(df_.set_index(["Technology", "country"]).variable_OM_Cost.to_dict(), "c_vom", gdx,
                  text="variable O&M cost [Euro per MWh]", overwrite=overwrite)                     
 
    # yearly CHP generation by technology and country
    gt.add_parameter(df_chp_gen.set_index(["technology", "country"]).MWh.to_dict(), "chp_gen", gdx,
                  text="yearly electricity generation from chp plants by country and technology [MWh]",
                  overwrite = overwrite)
    
    # yearly generation
    gt.add_parameter(df_annual.MWh.to_dict(), "gen_annual", gdx,
                  text="annual electricity generation by country and technology [MWh]",
                  overwrite = overwrite)    
    

Selection of dynamic parameters uses a dataframe with dates mapped to periods including the number of days the respective period occurs.

### Full dataset

In [119]:
def inject_dynamic(gdx, df_period):
    """ Injects time dependent parameters into gdx given period selection
    :param gdx: <gams.GamsDatabase> gdx to inject data
    :param df_periods: <pd.DataFrame> with mapping from dates to periods and lenght of respective period
    """
    # overwrite settings (False does not work due to bug in gdxtools)
    overwrite = True
    
    # extract mapping dates to periods
    map_periods = df_periods.period.to_dict()
    periods = map_periods.values()
    
    # sets
    gt.add_set(df_periods.period, "t", gdx, text="Periods", overwrite=overwrite)
    gt.add_set(df_periods.day, "d", gdx, text="days", overwrite=overwrite)
    gt.add_set(df_periods.month, "m", gdx, text="months", overwrite=overwrite)
    
    # period duration
    gt.add_parameter(df_period.reset_index().set_index("period").periodLength.to_dict(), "dur_d", gdx,
                  text="duration of days", overwrite=overwrite)
    
    # day mapping
    gt.add_set(list(df_periods.reset_index()[["period", "day"]].itertuples(index=False, name=None)),
               "map_t_d", gdx, text="mapping of periods to dates",
               overwrite=overwrite)
            
    # month mapping
    gt.add_set(list(df_periods.reset_index()[["period", "month"]].itertuples(index=False, name=None)),
               "map_t_m", gdx, text="mapping of periods to months",
               overwrite=overwrite)
            
    # extract and select time related parameters
    def extract_and_select(df):
        df_ = df.unstack().reset_index()
        df_["period"] = df_.date.map(map_periods)
        df_ = df_[df_.period.notnull()] 
        return df_
    
    # load parameter 
    df_ = df_load.reset_index()
    df_["period"] = df_.time.map(map_periods)
    df_ = df_[df_.period.notnull()]
    df_ = df_.set_index(["country", "period"])
    gt.add_parameter(df_.MWh.to_dict(), "demand", gdx,
                text="demand per period [MWh]", overwrite=overwrite)  
    
    # reference prices
    df_ = extract_and_select(df_price)
    gt.add_parameter(df_.set_index(["level_0","period"])[0].to_dict(), "pRef", gdx,
                text="hourly reference price [Euro per MWh]", overwrite=overwrite) 
    
    # monthly generation for availability calibration
    df_ = df_generation_monthly.reset_index()
    df_ = df_[df_.month.notnull()]
    df_ = df_.set_index(["technology", "country", "month"])
    gt.add_parameter(df_.MWh.to_dict(), "gen_monthly", gdx,
                text="monthly electricity generation [MWh]", overwrite=overwrite) 
    
    # monthly peakload availability from entsoe outage data
    df_ = df_avail_peak.reset_index()
    df_ = df_[df_.Month.notnull()]
    df_ = df_.set_index(["technology", "country", "Month"])
    gt.add_parameter(df_avail_peak.avail.to_dict(), "availpeakUp", gdx, 
                  text="peak load plant availability [%]", overwrite=overwrite)
        
    # renewable supply
    df_ = df_res.reset_index()
    df_["period"] = df_.time.map(map_periods)
    df_ = df_[df_.period.notnull()]
    df_ = df_.set_index(["technology", "country", "period"])
    gt.add_parameter(df_.profile.to_dict(), "renS", gdx,
                text="hourly renewable supply profile [% of yearly generation]", overwrite=overwrite)  
    
    # ntc values
    df_ = df_ntc.copy()
    df_["period"] = df_.date.map(map_periods)
    df_ = df_[df_.period.notnull()]
    gt.add_parameter(df_.set_index(["from", "to", "period"]).ntc.to_dict(), "ntc", gdx,
              text="maximum output change in terms of total capacity [%]", overwrite=True)    
    
    # run-of-river generation
    df_ = df_ror.reset_index().copy()
    df_["period"] = df_.time.map(map_periods)
    df_ = df_[df_.period.notnull()]
    gt.add_parameter(df_.set_index(["country", "period"]).MWh.to_dict(), "gen_ror_exog", gdx,
              text="exogenous hourly run-of-river generation [MWh]", overwrite=True)  
    
    # reservoir initial levels (normalized)
    df_ = df_reservoir_level_final.copy()
    df_["period"] = df_.date.map(map_periods)
    df_ = df_[df_.period.notnull()]
    gt.add_parameter(df_.set_index(["country", "period", "technology"]).MWh.to_dict(), "reservoir_initial_up", gdx,
              text="upload reservoir initial level [MWh]", overwrite=True) 
    
    # reservoir inflow
    #this is changed from relative to absolute values in comparison to old version!
    df_ = df_storage_inflows.copy()
    df_["period"] = df_.date.map(map_periods)
    df_ = df_[df_.period.notnull()]
    gt.add_parameter(df_.set_index(["country", "period", "technology"]).MWh.to_dict(), "reservoir_inflow_up", gdx,
              text="upload reservoir inflow [MWh]", overwrite=True) 
      
    # heat demand profile
    df_ = df_heat_dem_in.copy()
    df_["period"] = df_.date.map(map_periods)
    df_ = df_[df_.period.notnull()]
    gt.add_parameter(df_.set_index(["period"]).heat_demand_relative.to_dict(), "heat_dem_up", gdx,
              text = "upload normalized heat demand [%]", overwrite = True)

Create a mapping from dates to time period in the model:

In [120]:
df_periods = pd.Series({d: "t%04d" % (i+1) for i, d in enumerate(df_load.reset_index().time.unique())}).to_frame("period")
df_periods['day'] = df_periods.index.date
df_periods['month'] = df_periods.index.month
df_periods["periodLength"] = 1

In [121]:
df_periods.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 8760 entries, 2017-01-01 00:00:00 to 2017-12-31 23:00:00
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   period        8760 non-null   object
 1   day           8760 non-null   object
 2   month         8760 non-null   int64 
 3   periodLength  8760 non-null   int64 
dtypes: int64(2), object(2)
memory usage: 342.2+ KB


Create gdx and workspace

In [122]:
dir_gms = os.getcwd()
ws = gams.GamsWorkspace(dir_gms)
gdx_all = ws.add_database()

Inject to gdx file

In [123]:
inject_static(gdx_all)

In [124]:
inject_dynamic(gdx_all, df_periods)

Save gdx

In [125]:
fn_out = dir_gdx + "data" + countries_out + "_%d_all.gdx" % baseyear
gdx_all.export(fn_out)

Also save to drop box if set to yes

In [126]:
if dropbox == 'yes':
    fn_out = "C:/Users/jonas/Dropbox/eu_model_data/" + "data" + countries_out + "_%d_all.gdx" % baseyear
    gdx_all.export(fn_out)

### Calibration data

Static data plus the following dynamic parameters:
- r_price(country,t)
- r_demand(country,t)
- r_generation(tech,country,t)
- TRADE(fromcountry,tocountry,t)

In [127]:
def inject_static_calibration(gdx):
    """Injects static data into gdx file
    :param gdx: <gams.GamsDatabase> gdx to inject data
    """
    # overwrite settings (False does not work due to bug in gdxtools)
    overwrite = True
    
    # sets
    ## technology sets
    gt.add_set(sorted(countries), "c", gdx, text="countries", overwrite=overwrite)
    gt.add_set(technologies, "tech", gdx, text="technologies", overwrite=overwrite)
    gt.add_set(conventionals, "i", gdx, text="conventional technologies", overwrite=overwrite)
    gt.add_set(storages, "s", gdx, text="storage facilities", overwrite=overwrite)  
    gt.add_set(renewables, "r", gdx, text="new renewable sources", overwrite=overwrite)  
    gt.add_set(old_renewables, "ro", gdx, text="old = conventional renewable sources", overwrite=overwrite)  
    gt.add_set(all_renewables, "ra", gdx, text="all renewable sources", overwrite=overwrite)  
    gt.add_set(baseload, "baseload", gdx, text="baseload technologies excl. RoR", overwrite=overwrite)  
    gt.add_set(fixed_feedin, "fixed", gdx, text="fixed feedin technologies", overwrite=overwrite)  
    gt.add_set(peakload, "peak", gdx, text="peak load technologies", overwrite=overwrite)  


In [128]:
def inject_dynamic_calibration(gdx, df_period):
    """ Injects time dependent parameters into gdx given period selection
    :param gdx: <gams.GamsDatabase> gdx to inject data
    :param df_periods: <pd.DataFrame> with mapping from dates to periods and lenght of respective period
    """
    # overwrite settings (False does not work due to bug in gdxtools)
    overwrite = True
    
    # extract mapping dates to periods
    map_periods = df_periods.period.to_dict()
    periods = map_periods.values()
    
    # sets
    gt.add_set(df_periods.period, "t", gdx, text="Periods", overwrite=overwrite)
    gt.add_set(df_periods.day, "d", gdx, text="days", overwrite=overwrite)
    gt.add_set(df_periods.month, "m", gdx, text="months", overwrite=overwrite)
    
    # period duration
    gt.add_parameter(df_period.reset_index().set_index("period").periodLength.to_dict(), "dur_d", gdx,
                  text="duration of days", overwrite=overwrite)
    
    # day mapping
    gt.add_set(list(df_periods.reset_index()[["period", "day"]].itertuples(index=False, name=None)),
               "map_t_d", gdx, text="mapping of periods to dates",
               overwrite=overwrite)
    
    # month mapping
    gt.add_set(list(df_periods.reset_index()[["period", "month"]].itertuples(index=False, name=None)),
               "map_t_m", gdx, text="mapping of periods to months",
               overwrite=overwrite)
    
    # extract and select time related parameters
    def extract_and_select(df):
        df_ = df.unstack().reset_index()
        df_["period"] = df_.date.map(map_periods)
        df_ = df_[df_.period.notnull()] 
        return df_
    
    # prices
    df_ = extract_and_select(df_price)
    gt.add_parameter(df_.set_index(["level_0","period"])[0].to_dict(), "r_price", gdx,
                text="hourly reference price [Euro per MWh]", overwrite=overwrite) 
    
    # load 
    df_ = df_load.reset_index()
    df_["period"] = df_.time.map(map_periods)
    df_ = df_[df_.period.notnull()]
    df_ = df_.set_index(["country", "period"])
    gt.add_parameter(df_.MWh.to_dict(), "r_demand", gdx,
                text="demand per period [MWh]", overwrite=overwrite)  
    
    # generation
    df_ = df_generation.reset_index()
    df_["period"] = df_.time.map(map_periods)
    df_ = df_[df_.period.notnull()]
    df_ = df_.set_index(["technology", "country", "period"])
    gt.add_parameter(df_.MWh.to_dict(), "r_generation", gdx,
                text="hourly electricity generation [MWh]", overwrite=overwrite)  
    
    # trade
    df_ = df_trade.reset_index()
    df_["period"] = df_.time.map(map_periods)
    df_ = df_[df_.period.notnull()]
    df_ = df_.set_index(["from_country", "to_country", "period"])
    gt.add_parameter(df_.MWh.to_dict(), "TRADE", gdx,
                text="hourly cross-border trade [MWh]", overwrite=overwrite)   

Create gdx and workspace

In [129]:
gdx_calibration = ws.add_database()

Inject to gdx file

In [130]:
inject_static_calibration(gdx_calibration)

In [131]:
inject_dynamic_calibration(gdx_calibration, df_periods)

Save gdx

In [132]:
fn_out = dir_gdx + "data" + countries_out + "_%d_all_calibration.gdx" % baseyear
gdx_calibration.export(fn_out)

Also export to dropbox

In [133]:
if dropbox == 'yes':
    fn_out = "C:/Users/jonas/Dropbox/eu_model_data/" + "data" + countries_out + "_%d_all_calibration.gdx" % baseyear
    gdx_calibration.export(fn_out)

### Short data set: One day per month


Create gdx file

In [134]:
gdx_short = ws.add_database()

Period settings  
Here, we use 24 hours of the 15th for each month

In [135]:
dates = []
periodLength = []
for m in range(1,13):
    start_date = datetime.datetime(baseyear, m, 15, 0,0)
    end_date = datetime.datetime(baseyear, m, 15, 23,0)
    dates.extend(pd.date_range(start_date, end_date, freq="H"))
    _, l = calendar.monthrange(baseyear, m)
    periodLength.extend([l]*24)
p = ["t%04d" % (i+1) for i in range(0, len(dates))]
df_periods = pd.DataFrame({"date": dates,"periodLength": periodLength, "period": p}).set_index("date")
df_periods['day'] = df_periods.index.date
df_periods['month'] = df_periods.index.month
df_periods.head(1)

,periodLength,period,day,month
date,,,,
2017-01-15,31,t0001,2017-01-15,1


In [136]:
inject_static(gdx_short)

In [137]:
inject_dynamic(gdx_short, df_periods)

Save gdx

In [138]:
fn_out = dir_gdx + "data" + countries_out + "_%d_15.gdx" % baseyear
gdx_short.export(fn_out)

and calibration

In [139]:
gdx_calibration_short = ws.add_database()

In [140]:
inject_static_calibration(gdx_calibration_short)

In [141]:
inject_dynamic_calibration(gdx_calibration_short, df_periods)

In [142]:
fn_out = dir_gdx + "data" + countries_out + "_%d_15_calibration.gdx" % baseyear
gdx_calibration_short.export(fn_out)

Also export to dropbox

In [143]:
if dropbox == 'yes':
    fn_out = "C:/Users/jonas/Dropbox/eu_model_data/" + "data" + countries_out + "_%d_15.gdx" % baseyear
    gdx_short.export(fn_out)

In [144]:
if dropbox == 'yes':
    fn_out = "C:/Users/jonas/Dropbox/eu_model_data/" + "data" + countries_out + "_%d_15_calibration.gdx" % baseyear
    gdx_calibration_short.export(fn_out)